In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
data=pd.read_csv("D:\Customer-Segmentation-RFM\Online Retail.csv")
print(data.head())
print(data.tail())
print(data.info())
print(data.isnull().sum())

<>:1: SyntaxWarning: invalid escape sequence '\C'
<>:1: SyntaxWarning: invalid escape sequence '\C'
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_14836\1291124564.py:1: SyntaxWarning: invalid escape sequence '\C'
  data=pd.read_csv("D:\Customer-Segmentation-RFM\Online Retail.csv")


  InvoiceNo InvoiceDate  InvoiceTime StockCode  \
0    536365  01-12-2010  08:26:00 AM    85123A   
1    536365  01-12-2010  08:26:00 AM     71053   
2    536365  01-12-2010  08:26:00 AM    84406B   
3    536365  01-12-2010  08:26:00 AM    84029G   
4    536365  01-12-2010  08:26:00 AM    84029E   

                           Description  Quantity  UnitPrice  Totalsale  \
0   WHITE HANGING HEART T-LIGHT HOLDER         6       2.55      15.30   
1                  WHITE METAL LANTERN         6       3.39      20.34   
2       CREAM CUPID HEARTS COAT HANGER         8       2.75      22.00   
3  KNITTED UNION FLAG HOT WATER BOTTLE         6       3.39      20.34   
4       RED WOOLLY HOTTIE WHITE HEART.         6       3.39      20.34   

   CustomerID         Country  
0     17850.0  United Kingdom  
1     17850.0  United Kingdom  
2     17850.0  United Kingdom  
3     17850.0  United Kingdom  
4     17850.0  United Kingdom  
       InvoiceNo InvoiceDate  InvoiceTime StockCode  \
541904 

In [4]:
# Remove Missing Customer ID
data=data.dropna(subset=['CustomerID'])

#Convert date column
data['InvoiceDate']=pd.to_datetime(data['InvoiceDate'], dayfirst=True)

#Create TotalPrice Column
data['TotalPrice']=data['Quantity'] * data['UnitPrice']

In [5]:
reference_date=data['InvoiceDate'].max() + pd.Timedelta(days=1)

In [9]:
rfm=data.groupby('CustomerID').agg({
    'InvoiceDate':lambda x: (reference_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'})

rfm.columns=['Recency', 'Frequency', 'Monetary']

In [11]:
rfm['R_score']=pd.qcut(rfm['Recency'], 4, labels=[4,3,2,1])
rfm['F_score']=pd.qcut(rfm['Frequency'].rank(method='first'), 4,labels=[1,2,3,4])
rfm['M_score']=pd.qcut(rfm['Monetary'],4,labels=[1,2,3,4])
rfm['RFM_Score']=rfm['R_score'].astype(str) + rfm['F_score'].astype(str)+ rfm['M_score'].astype(str)

In [12]:
#Convert scores to integers
rfm['R_score'] = rfm['R_score'].astype(int)
rfm['F_score'] = rfm['F_score'].astype(int)
rfm['M_score'] = rfm['M_score'].astype(int)

#Define segmentation function
def segment(row):
    if row['R_score'] == 4 and row['F_score'] == 4 and row['M_score'] >= 3:
        return 'VIP'
    elif row['R_score'] >= 3 and row['F_score'] >= 3:
        return 'Loyal'
    elif row['R_score'] <= 2 and row['F_score'] <= 2:
        return 'At Risk'
    else:
        return 'Regular'

#Apply function to each row
rfm['Segment'] = rfm.apply(segment, axis=1)

In [ ]:
print(rfm.head())